In [1]:
import pandas as pd

In [2]:
PATH = 'data/ContingencyDataASARCO.csv'
df_complete = pd.read_csv(PATH, parse_dates=['timestamp'])

In [3]:
OFFSET_HOURS = 3
df_complete['timestamp'] = df_complete['timestamp'] + pd.Timedelta(unit='hours', value=OFFSET_HOURS)

In [4]:
group_asarco = ['stop_reason', 'timecat']
resume_activities = df_complete[group_asarco].groupby(by=group_asarco).\
                                              agg({'timecat': 'count'}).\
                                              rename(columns={'timecat': 'count'}).\
                                              reset_index().\
                                              sort_values('count', ascending=False)

In [5]:
resume_activities.groupby(by=['timecat'])['timecat'].count().rename('count')

timecat
Accidente        2
Demora Nprog    13
Demora Prog      5
Efectivo         2
MARC             9
Mant Nprog      11
Mant Prog        9
Reserva          9
Name: count, dtype: int64

In [6]:
# MARC: Maintenance and Repair Contract
resume_activities[resume_activities['timecat'] == 'MARC']

,stop_reason,timecat,count
54,Sin infraestructura de apoyo,MARC,103
23,INSTALACIÓN SISTEMA TECNOLÓGI,MARC,22
51,SISTEMA MESH COASIN,MARC,19
52,SMARTCAP,MARC,9
55,Sistema contra incendios,MARC,7
42,Radiocomunicacion,MARC,6
50,SISTEMA CAS,MARC,6
56,Sistema dispatch - modular,MARC,2
11,Cambio vidrios,MARC,1


In [8]:
import plotly.express as px

In [18]:
df_translation = pd.read_csv('data/mapped_stop_reasons.csv')

In [34]:
resume_activities_asarco = resume_activities.copy()
asarco_activities = resume_activities['timecat'].unique()
asarco_values = ['Efectivo', 'Demora No Prog', 'Demora Prog', 'Mant No Prog', 'Reserva', 'Mant Prog', 'MARC', 'Accidente']
asarco_mapper = dict(zip(asarco_activities, asarco_values))
resume_activities_asarco['timecat'] = resume_activities_asarco['timecat'].map(asarco_mapper)
resume_activities_asarco = pd.merge(df_translation[['stop_reason', 'manual_parsing']],\
                                    resume_activities_asarco,\
                                    left_on='stop_reason',\
                                    right_on='stop_reason')
resume_activities_asarco = resume_activities_asarco.drop(columns='stop_reason').\
                                                    rename(columns={'manual_parsing': 'event', 'timecat': 'asarco_event'})

In [35]:
resume_activities_asarco

,event,asarco_event,count
0,Colacion en cabina 1,Demora Prog,1253
1,Colacion en cabina 2,Demora Prog,1076
2,Cambio de turno,Demora Prog,4192
3,Carga de combustible,Demora No Prog,1891
4,Colacion en comedor,Demora Prog,1047
5,Detenido por somnolencia,Demora No Prog,68
6,Espera de combustible,Demora No Prog,523
7,Evacuacion por tronadura,Demora No Prog,172
8,Evento geotecnico,Demora No Prog,20
9,Mina en emergencia,Demora No Prog,27


In [41]:
df_sunburst = resume_activities_asarco[['asarco_event', 'event']].reset_index(drop=True)
df_sunburst['values'] = 1

# Plot the tree
fig = px.sunburst(df_sunburst, path=['asarco_event', 'event'], values='values')
fig.update_layout(font=dict(size=30))
FOR_SAVE = True
if FOR_SAVE:
  SAVE_PATH = 'images/ASARCO_map.png'
  SAVE_SPECS = {'width': 900, 'height': 1000, 'scale': 1}
  fig.write_image(SAVE_PATH, **SAVE_SPECS)

fig